In [8]:

from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch

# 1. Reproducibility
SEED = 7

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 2. Locate project root
PROJECT_ROOT = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "experiments" / "04_electricity").is_dir()
        and (p / "datasets" / "electricity").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate predictive-analytics project root"
    )

# 3. Locate raw and preprocessed data
RAW_PATH = (
    PROJECT_ROOT / "datasets" / "electricity"
    / "energy_dataset.csv"
)

BUNDLE_PATH = (
    PROJECT_ROOT / "experiments" / "04_electricity"
    / "data" / "electricity_train_test.pt"
)

assert RAW_PATH.is_file()
assert BUNDLE_PATH.is_file()

# 4. Load existing preprocessed dataset
bundle = torch.load(
    BUNDLE_PATH,
    map_location="cpu",
    weights_only=True
)

train_data = bundle["train"]
test_data = bundle["test"]
channels = bundle["channels"]

# 5. Validate data
assert train_data.shape == (2800, 6)
assert test_data.shape == (700, 6)
assert len(channels) == 6

assert torch.isfinite(train_data).all()
assert torch.isfinite(test_data).all()

# 6. Display information
print("=== DEEPKOOPFORMER ABLATION ===")
print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

print("\nDataset: Electricity")
print("Train shape:", tuple(train_data.shape))
print("Test shape:", tuple(test_data.shape))
print("Channels:", channels)

print("\nSource indices:",
      bundle["source_start_idx"],
      bundle["source_end_idx"])

print("Missing strategy:", bundle["missing_strategy"])
print("Data SHA256:", bundle["data_hash"])

print("\nCell 1 validation passed!")


=== DEEPKOOPFORMER ABLATION ===
Project root: c:\Users\Win 11 New\Desktop\pa gk
Device: cuda
PyTorch: 2.0.1+cu118

Dataset: Electricity
Train shape: (2800, 6)
Test shape: (700, 6)
Channels: ['generation biomass', 'generation fossil brown coal/lignite', 'generation fossil gas', 'generation fossil hard coal', 'generation fossil oil', 'generation geothermal']

Source indices: 4000 7500
Missing strategy: forward_fill
Data SHA256: c024e2bb9460bf6e35d6f3353b82b6d90bee422366be0d6b3bb769a340eb0f0f

Cell 1 validation passed!


In [9]:

from sklearn.preprocessing import MinMaxScaler
import hashlib

# 1. Read original electricity dataset
df_raw = pd.read_csv(RAW_PATH)

START_IDX = bundle["source_start_idx"]
END_IDX = bundle["source_end_idx"]

# 2. Select the same 3500 timestamps and 6 channels
electricity_df = df_raw.iloc[
    START_IDX:END_IDX
][channels].copy()

print("=== ORIGINAL DATA ===")
print("Original dataset shape:", df_raw.shape)
print("Selected data shape:", electricity_df.shape)

assert electricity_df.shape == (3500, 6)

# 3. Check missing values
missing_mask = electricity_df.isna().any(axis=1)

print("\nMissing rows:", missing_mask.sum())
print("Missing values per channel:")
print(electricity_df.isna().sum())

# 4. Apply the preprocessing used previously
electricity_clean = electricity_df.ffill()

electricity_raw = electricity_clean.to_numpy(
    dtype=np.float32
)

assert np.isfinite(electricity_raw).all()

# 5. Verify original data hash
raw_hash = hashlib.sha256(
    electricity_raw.tobytes()
).hexdigest()

print("\n=== DATA HASH VALIDATION ===")
print("Reconstructed:", raw_hash)
print("Saved:        ", bundle["data_hash"])

assert raw_hash == bundle["data_hash"]

# 6. Chronological Train/Test Split
TRAIN_RATIO = bundle["train_ratio"]

split_idx = int(len(electricity_raw) * TRAIN_RATIO)

train_raw = electricity_raw[:split_idx]
test_raw = electricity_raw[split_idx:]


# 7. Validate Train/Test shapes
assert train_raw.shape == (2800, 6)
assert test_raw.shape == (700, 6)

# 8. Min-Max Scaling (fit on Train only)
scaler = MinMaxScaler(feature_range=(0, 1))

train_scaled = scaler.fit_transform(
    train_raw
).astype(np.float32)

test_scaled = scaler.transform(
    test_raw
).astype(np.float32)

# 9. Compare with previously saved tensors
saved_train = train_data.cpu().numpy()
saved_test = test_data.cpu().numpy()

train_diff = np.max(
    np.abs(train_scaled - saved_train)
)

test_diff = np.max(
    np.abs(test_scaled - saved_test)
)

np.testing.assert_allclose(
    train_scaled,
    saved_train,
    rtol=1e-6,
    atol=1e-6
)

np.testing.assert_allclose(
    test_scaled,
    saved_test,
    rtol=1e-6,
    atol=1e-6
)

# 10. Final validation
assert np.isfinite(train_scaled).all()
assert np.isfinite(test_scaled).all()

print("\n=== ELECTRICITY DATA VALIDATION ===")
print("Train:", train_scaled.shape)
print("Test:", test_scaled.shape)
print("Maximum Train difference:", train_diff)
print("Maximum Test difference:", test_diff)

print("\nData hash matched.")
print("Min-Max Scaling validated.")
print("Electricity preprocessing validation passed!")


=== ORIGINAL DATA ===
Original dataset shape: (35064, 29)
Selected data shape: (3500, 6)

Missing rows: 1
Missing values per channel:
generation biomass                      1
generation fossil brown coal/lignite    1
generation fossil gas                   1
generation fossil hard coal             1
generation fossil oil                   1
generation geothermal                   1
dtype: int64

=== DATA HASH VALIDATION ===
Reconstructed: c024e2bb9460bf6e35d6f3353b82b6d90bee422366be0d6b3bb769a340eb0f0f
Saved:         c024e2bb9460bf6e35d6f3353b82b6d90bee422366be0d6b3bb769a340eb0f0f

=== ELECTRICITY DATA VALIDATION ===
Train: (2800, 6)
Test: (700, 6)
Maximum Train difference: 0.0
Maximum Test difference: 0.0

Data hash matched.
Min-Max Scaling validated.
Electricity preprocessing validation passed!


Dataset Inventory

In [10]:
# 1. Locate all four ablation datasets
ABLATION_FILES = {
    "Electricity": (
        PROJECT_ROOT / "experiments" / "04_electricity"
        / "data" / "electricity_train_test.pt"
    ),
    "Financial": (
        PROJECT_ROOT / "experiments" / "03_financial"
        / "data" / "financial_train_test.pt"
    ),
    "CMIP6 Wind": (
        PROJECT_ROOT / "experiments" / "02_climate"
        / "data" / "scenario4_wind_train_test.pt"
    ),
    "CMIP6 Pressure": (
        PROJECT_ROOT / "experiments" / "02_climate"
        / "data" / "pressure_nearest_5sites.pt"
    ),
}

EXPECTED_SHAPES = {
    "Electricity": ((2800, 6), (700, 6)),
    "Financial": ((2320, 2), (580, 2)),
    "CMIP6 Wind": ((2336, 5), (584, 5)),
    "CMIP6 Pressure": ((400, 5), None),
}

# 2. Load datasets
ABLATION_DATASETS = {}
inventory_rows = []

for name, path in ABLATION_FILES.items():

    print(f"\n{'=' * 60}")
    print("DATASET:", name)
    print("File:", path)

    if not path.is_file():
        raise FileNotFoundError(
            f"Dataset file not found: {path}"
        )

    # Load trusted files created in our project.
    saved = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    print("Available keys:", list(saved.keys()))

    # 3. Identify stored data format
    if name == "CMIP6 Pressure":

        raw = saved["pressure_raw"]
        train_array = None
        test_array = None

    else:

        train_key = (
            "train" if "train" in saved
            else "train_scaled"
        )

        test_key = (
            "test" if "test" in saved
            else "test_scaled"
        )

        train_array = saved[train_key]
        test_array = saved[test_key]
        raw = None

    # 4. Convert to NumPy for validation
    def to_numpy(value):
        if isinstance(value, torch.Tensor):
            return value.detach().cpu().numpy()
        return np.asarray(value)

    if raw is not None:

        raw = to_numpy(raw)

        assert raw.ndim == 2
        assert np.isfinite(raw).all()

        actual_shape = tuple(raw.shape)
        expected_raw, _ = EXPECTED_SHAPES[name]

        assert actual_shape == expected_raw

        n_time, n_channels = raw.shape
        train_shape = None
        test_shape = None

        print("Raw shape:", actual_shape)
        print("Split status: NOT YET SPLIT")

    else:

        train_array = to_numpy(train_array)
        test_array = to_numpy(test_array)

        assert train_array.ndim == 2
        assert test_array.ndim == 2

        assert np.isfinite(train_array).all()
        assert np.isfinite(test_array).all()

        train_shape = tuple(train_array.shape)
        test_shape = tuple(test_array.shape)

        expected_train, expected_test = EXPECTED_SHAPES[name]

        assert train_shape == expected_train
        assert test_shape == expected_test

        assert train_array.shape[1] == test_array.shape[1]

        n_time = len(train_array) + len(test_array)
        n_channels = train_array.shape[1]

        print("Train shape:", train_shape)
        print("Test shape:", test_shape)
        print("Split status: AVAILABLE")

    # 5. Store in memory for subsequent data checks
    ABLATION_DATASETS[name] = {
        "path": path,
        "metadata": saved,
        "raw": raw,
        "train": train_array,
        "test": test_array,
    }

    inventory_rows.append({
        "Dataset": name,
        "Timestamps": n_time,
        "Channels": n_channels,
        "Train": str(train_shape),
        "Test": str(test_shape),
        "FileSizeMB": round(
            path.stat().st_size / (1024 ** 2), 3
        ),
        "Finite": True
    })

# 6. Display summary
inventory_df = pd.DataFrame(inventory_rows)

print("\n=== ABLATION DATASET INVENTORY ===")
display(inventory_df)

assert len(ABLATION_DATASETS) == 4

print("\nAll 4 dataset files loaded and validated!")
print("Note: Pressure Train/Test split is still pending.")



DATASET: Electricity
File: c:\Users\Win 11 New\Desktop\pa gk\experiments\04_electricity\data\electricity_train_test.pt
Available keys: ['train', 'test', 'channels', 'data_hash', 'train_ratio', 'source_start_idx', 'source_end_idx', 'missing_strategy', 'scaler_min', 'scaler_max']
Train shape: (2800, 6)
Test shape: (700, 6)
Split status: AVAILABLE

DATASET: Financial
File: c:\Users\Win 11 New\Desktop\pa gk\experiments\03_financial\data\financial_train_test.pt
Available keys: ['train', 'test', 'channels', 'input_dim', 'train_ratio', 'train_size', 'test_size', 'scaler_data_min', 'scaler_data_max', 'data_hash']
Train shape: (2320, 2)
Test shape: (580, 2)
Split status: AVAILABLE

DATASET: CMIP6 Wind
File: c:\Users\Win 11 New\Desktop\pa gk\experiments\02_climate\data\scenario4_wind_train_test.pt
Available keys: ['train_scaled', 'test_scaled', 'train_raw', 'test_raw', 'location_indices', 'scaler_min', 'scaler_max', 'seq_len', 'patch_lengths', 'horizons', 'd_model', 'epochs', 'learning_rate', '

,Dataset,Timestamps,Channels,Train,Test,FileSizeMB,Finite
0,Electricity,3500,6,"(2800, 6)","(700, 6)",0.082,True
1,Financial,2900,2,"(2320, 2)","(580, 2)",0.024,True
2,CMIP6 Wind,2920,5,"(2336, 5)","(584, 5)",0.114,True
3,CMIP6 Pressure,400,5,None,None,0.009,True



All 4 dataset files loaded and validated!
Note: Pressure Train/Test split is still pending.


Electricity Dataset Validation

In [11]:
from sklearn.preprocessing import MinMaxScaler
import hashlib

# 1. Read original electricity dataset
df_raw = pd.read_csv(RAW_PATH)

START_IDX = bundle["source_start_idx"]
END_IDX = bundle["source_end_idx"]

# 2. Select the same 3500 timestamps and 6 channels
electricity_df = df_raw.iloc[
    START_IDX:END_IDX
][channels].copy()

print("=== ORIGINAL DATA ===")
print("Original dataset shape:", df_raw.shape)
print("Selected data shape:", electricity_df.shape)

assert electricity_df.shape == (3500, 6)

# 3. Check missing values
missing_mask = electricity_df.isna().any(axis=1)

print("\nMissing rows:", missing_mask.sum())
print("Missing values per channel:")
print(electricity_df.isna().sum())

# 4. Apply the preprocessing used previously
electricity_clean = electricity_df.ffill()

electricity_raw = electricity_clean.to_numpy(
    dtype=np.float32
)

assert np.isfinite(electricity_raw).all()

# 5. Verify original data hash
raw_hash = hashlib.sha256(
    electricity_raw.tobytes()
).hexdigest()

print("\n=== DATA HASH VALIDATION ===")
print("Reconstructed:", raw_hash)
print("Saved:        ", bundle["data_hash"])

assert raw_hash == bundle["data_hash"]

# 6. Chronological Train/Test Split
TRAIN_RATIO = bundle["train_ratio"]

split_idx = int(len(electricity_raw) * TRAIN_RATIO)

train_raw = electricity_raw[:split_idx]
test_raw = electricity_raw[split_idx:]

assert train_raw.shape == (2800, 6)
assert test_raw.shape == (700, 6)

# 7. Min-Max Scaling — fit on Train only
scaler = MinMaxScaler(feature_range=(0, 1))

train_scaled = scaler.fit_transform(
    train_raw
).astype(np.float32)

test_scaled = scaler.transform(
    test_raw
).astype(np.float32)

# 8. Compare with previously saved tensors
saved_train = train_data.cpu().numpy()
saved_test = test_data.cpu().numpy()

train_diff = np.max(
    np.abs(train_scaled - saved_train)
)

test_diff = np.max(
    np.abs(test_scaled - saved_test)
)

np.testing.assert_allclose(
    train_scaled,
    saved_train,
    rtol=1e-6,
    atol=1e-6
)

np.testing.assert_allclose(
    test_scaled,
    saved_test,
    rtol=1e-6,
    atol=1e-6
)

# 9. Validate scaler statistics
np.testing.assert_allclose(
    scaler.data_min_,
    bundle["scaler_min"]
)

np.testing.assert_allclose(
    scaler.data_max_,
    bundle["scaler_max"]
)

# 10. Numerical validation
assert np.isfinite(train_scaled).all()
assert np.isfinite(test_scaled).all()

# 11. Update ablation dataset inventory
ABLATION_DATASETS["Electricity"]["raw"] = electricity_raw
ABLATION_DATASETS["Electricity"]["train"] = train_scaled
ABLATION_DATASETS["Electricity"]["test"] = test_scaled

# 12. Final summary
print("\n=== ELECTRICITY DATA VALIDATION ===")
print("Original shape:", electricity_raw.shape)
print("Train:", train_scaled.shape)
print("Test:", test_scaled.shape)

print("\nMaximum Train difference:", train_diff)
print("Maximum Test difference:", test_diff)

print("\nScaler fitted on Train only.")
print("Scaler statistics matched.")
print("Data hash matched.")
print("Electricity preprocessing validation passed!")

=== ORIGINAL DATA ===
Original dataset shape: (35064, 29)
Selected data shape: (3500, 6)

Missing rows: 1
Missing values per channel:
generation biomass                      1
generation fossil brown coal/lignite    1
generation fossil gas                   1
generation fossil hard coal             1
generation fossil oil                   1
generation geothermal                   1
dtype: int64

=== DATA HASH VALIDATION ===
Reconstructed: c024e2bb9460bf6e35d6f3353b82b6d90bee422366be0d6b3bb769a340eb0f0f
Saved:         c024e2bb9460bf6e35d6f3353b82b6d90bee422366be0d6b3bb769a340eb0f0f

=== ELECTRICITY DATA VALIDATION ===
Original shape: (3500, 6)
Train: (2800, 6)
Test: (700, 6)

Maximum Train difference: 0.0
Maximum Test difference: 0.0

Scaler fitted on Train only.
Scaler statistics matched.
Data hash matched.
Electricity preprocessing validation passed!


FINANCIAL DATASET VALIDATION

In [12]:
# 1. Locate Financial dataset files
FINANCIAL_DIR = (
    PROJECT_ROOT / "experiments" / "03_financial" / "data"
)

FINANCIAL_CSV = FINANCIAL_DIR / "crypto_coins.csv"
FINANCIAL_NPY = FINANCIAL_DIR / "crypto_coins.npy"

assert FINANCIAL_CSV.is_file()
assert FINANCIAL_NPY.is_file()

# 2. Load original Financial dataset
fin_df = pd.read_csv(FINANCIAL_CSV)

fin_record = ABLATION_DATASETS["Financial"]
fin_bundle = fin_record["metadata"]

print("=== FINANCIAL ORIGINAL DATA ===")
print("Original shape:", fin_df.shape)
print("Columns:", fin_df.columns.tolist())

assert {"Symbol", "ClosePrice", "CloseTime"}.issubset(
    fin_df.columns
)

# 3. Reconstruct the Financial data used previously
fin_close_prices = fin_df["ClosePrice"].to_numpy(
    dtype=np.float32
)

FIN_SEGMENT_LEN = 2900
FIN_SELECTED_ROWS = 5800

fin_selected = fin_close_prices[:FIN_SELECTED_ROWS]

assert len(fin_selected) == FIN_SELECTED_ROWS
assert fin_df["Symbol"].iloc[:FIN_SELECTED_ROWS].eq("BTC").all()

fin_raw = np.column_stack([
    fin_selected[:FIN_SEGMENT_LEN],
    fin_selected[FIN_SEGMENT_LEN:FIN_SELECTED_ROWS]
]).astype(np.float32)

assert fin_raw.shape == (2900, 2)
assert np.isfinite(fin_raw).all()

# 4. Verify saved NumPy data
fin_saved_npy = np.load(FINANCIAL_NPY)

np.testing.assert_array_equal(
    fin_raw,
    fin_saved_npy
)

# 5. Inspect timestamps of both channels
fin_time_1 = pd.to_datetime(
    fin_df["CloseTime"].iloc[:2900]
).reset_index(drop=True)

fin_time_2 = pd.to_datetime(
    fin_df["CloseTime"].iloc[2900:5800]
).reset_index(drop=True)

timestamps_aligned = fin_time_1.equals(fin_time_2)

print("\n=== FINANCIAL CHANNELS ===")
print("Data shape:", fin_raw.shape)
print("Channels:", fin_bundle["channels"])

print("Segment 1:",
      fin_time_1.iloc[0], "->", fin_time_1.iloc[-1])

print("Segment 2:",
      fin_time_2.iloc[0], "->", fin_time_2.iloc[-1])

print("Timestamps identical:", timestamps_aligned)

# 6. Chronological Train/Test split
FIN_TRAIN_RATIO = 0.8

fin_split_idx = int(
    len(fin_raw) * FIN_TRAIN_RATIO
)

fin_train_raw = fin_raw[:fin_split_idx]
fin_test_raw = fin_raw[fin_split_idx:]

assert fin_train_raw.shape == (2320, 2)
assert fin_test_raw.shape == (580, 2)

# 7. Min-Max Scaling (fit on Train only)
fin_scaler = MinMaxScaler(feature_range=(0, 1))

fin_train_scaled = fin_scaler.fit_transform(
    fin_train_raw
).astype(np.float32)

fin_test_scaled = fin_scaler.transform(
    fin_test_raw
).astype(np.float32)

# 8. Compare with saved Train/Test arrays
fin_saved_train = fin_record["train"]
fin_saved_test = fin_record["test"]

np.testing.assert_allclose(
    fin_train_scaled,
    fin_saved_train,
    rtol=1e-6,
    atol=1e-6
)

np.testing.assert_allclose(
    fin_test_scaled,
    fin_saved_test,
    rtol=1e-6,
    atol=1e-6
)

# 9. Validate scaler statistics
np.testing.assert_allclose(
    fin_scaler.data_min_,
    fin_bundle["scaler_data_min"]
)

np.testing.assert_allclose(
    fin_scaler.data_max_,
    fin_bundle["scaler_data_max"]
)

# 10. Verify SHA256
fin_hash = hashlib.sha256(
    fin_train_scaled.tobytes()
    + fin_test_scaled.tobytes()
).hexdigest()

assert fin_hash == fin_bundle["data_hash"]

# 11. Numerical validation
assert np.isfinite(fin_train_scaled).all()
assert np.isfinite(fin_test_scaled).all()

train_diff = np.max(
    np.abs(fin_train_scaled - fin_saved_train)
)

test_diff = np.max(
    np.abs(fin_test_scaled - fin_saved_test)
)

# 12. Update dataset inventory
fin_record["raw"] = fin_raw
fin_record["train"] = fin_train_scaled
fin_record["test"] = fin_test_scaled

# 13. Final summary
print("\n=== FINANCIAL DATA VALIDATION ===")
print("Original shape:", fin_raw.shape)
print("Train:", fin_train_scaled.shape)
print("Test:", fin_test_scaled.shape)

print("\nMaximum Train difference:", train_diff)
print("Maximum Test difference:", test_diff)

print("\nReconstructed SHA256:", fin_hash)
print("Saved SHA256:        ", fin_bundle["data_hash"])

print("\nNumPy data matched.")
print("Scaler statistics matched.")
print("Financial preprocessing validation passed!")


=== FINANCIAL ORIGINAL DATA ===
Original shape: (27585, 10)
Columns: ['CloseTime', 'OpenPrice', 'HighPrice', 'LowPrice', 'ClosePrice', 'volume', 'MarketCap', 'Symbol', 'Name', 'Date']

=== FINANCIAL CHANNELS ===
Data shape: (2900, 2)
Channels: ['BTC_segment_1', 'BTC_segment_2']
Segment 1: 2018-09-01 00:00:00 -> 2020-08-25 18:00:00
Segment 2: 2020-08-26 00:00:00 -> 2022-08-20 18:00:00
Timestamps identical: False

=== FINANCIAL DATA VALIDATION ===
Original shape: (2900, 2)
Train: (2320, 2)
Test: (580, 2)

Maximum Train difference: 0.0
Maximum Test difference: 0.0

Reconstructed SHA256: 9383a90ba5109d569c2f965b9698120f893a86f146efd05f7585659a3913f519
Saved SHA256:         9383a90ba5109d569c2f965b9698120f893a86f146efd05f7585659a3913f519

NumPy data matched.
Scaler statistics matched.
Financial preprocessing validation passed!


CMIP6 WIND SPEED DATASET VALIDATION

In [13]:
import xarray as xr
import hashlib

# 1. Locate original and saved datasets
WIND_NC_PATH = (
    PROJECT_ROOT / "datasets" / "climate" / "raw"
    / "Klima_Daten_10m_3h_2020_RCP26.nc"
)

CLIMATE_DATA_DIR = (
    PROJECT_ROOT / "experiments" / "02_climate" / "data"
)

WIND_RAW_PT = (
    CLIMATE_DATA_DIR / "wind_2920_5sites.pt"
)

PRESSURE_PT = (
    CLIMATE_DATA_DIR / "pressure_nearest_5sites.pt"
)

for path in [WIND_NC_PATH, WIND_RAW_PT, PRESSURE_PT]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing source file: {path}")

# 2. Load previous datasets and metadata
wind_record = ABLATION_DATASETS["CMIP6 Wind"]
wind_bundle = wind_record["metadata"]

wind_source_bundle = torch.load(
    WIND_RAW_PT, map_location="cpu", weights_only=False
)

pressure_bundle = torch.load(
    PRESSURE_PT, map_location="cpu", weights_only=False
)

def wind_to_numpy(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)

# 3. Verify the five spatial locations
grid_rows = np.asarray(
    pressure_bundle["grid_rows"], dtype=int
)

grid_cols = np.asarray(
    pressure_bundle["grid_cols"], dtype=int
)

location_indices = pressure_bundle["location_indices"]

assert len(grid_rows) == 5
assert len(grid_cols) == 5
assert len(location_indices) == 5

np.testing.assert_array_equal(
    grid_rows,
    wind_source_bundle["grid_rows"]
)

np.testing.assert_array_equal(
    grid_cols,
    wind_source_bundle["grid_cols"]
)

# 4. Reconstruct Wind Speed from original NetCDF
with xr.open_dataset(WIND_NC_PATH) as ds:

    assert "sfcWind" in ds.data_vars
    assert ds.sizes["time"] >= 2920

    wind_selected = ds["sfcWind"].isel(
        time=slice(0, 2920),
        rlat=xr.DataArray(grid_rows, dims="site"),
        rlon=xr.DataArray(grid_cols, dims="site")
    )

    wind_raw = (
        wind_selected
        .transpose("time", "site")
        .values
        .astype(np.float32)
    )

    wind_units = ds["sfcWind"].attrs.get(
        "units", "unknown"
    )

    start_time = ds["time"].values[0]
    end_time = ds["time"].values[2919]

assert wind_raw.shape == (2920, 5)
assert np.isfinite(wind_raw).all()

# 5. Verify the previously extracted raw data
saved_wind_raw = wind_to_numpy(
    wind_source_bundle["wind_raw"]
)

np.testing.assert_array_equal(
    wind_raw, saved_wind_raw
)

# 6. Chronological Train/Test split
WIND_TRAIN_RATIO = 0.8

wind_split_idx = int(
    len(wind_raw) * WIND_TRAIN_RATIO
)

wind_train_raw = wind_raw[:wind_split_idx]
wind_test_raw = wind_raw[wind_split_idx:]

assert wind_train_raw.shape == (2336, 5)
assert wind_test_raw.shape == (584, 5)

assert wind_source_bundle["train_split_index"] == wind_split_idx

# 7. Verify raw Train/Test stored in scenario4 bundle
np.testing.assert_array_equal(
    wind_train_raw,
    wind_to_numpy(wind_bundle["train_raw"])
)

np.testing.assert_array_equal(
    wind_test_raw,
    wind_to_numpy(wind_bundle["test_raw"])
)

# 8. Reconstruct Min-Max Scaling
wind_scaler = MinMaxScaler(feature_range=(0, 1))

wind_train_scaled = wind_scaler.fit_transform(
    wind_train_raw
).astype(np.float32)

wind_test_scaled = wind_scaler.transform(
    wind_test_raw
).astype(np.float32)

# 9. Compare against saved scaled datasets
saved_wind_train = wind_record["train"]
saved_wind_test = wind_record["test"]

np.testing.assert_allclose(
    wind_train_scaled, saved_wind_train,
    rtol=1e-6, atol=1e-6
)

np.testing.assert_allclose(
    wind_test_scaled, saved_wind_test,
    rtol=1e-6, atol=1e-6
)

# 10. Verify scaler statistics
np.testing.assert_allclose(
    wind_scaler.data_min_,
    wind_to_numpy(wind_bundle["scaler_min"]),
    rtol=1e-6, atol=1e-6
)

np.testing.assert_allclose(
    wind_scaler.data_max_,
    wind_to_numpy(wind_bundle["scaler_max"]),
    rtol=1e-6, atol=1e-6
)

# 11. Verify numerical validity
assert np.isfinite(wind_train_scaled).all()
assert np.isfinite(wind_test_scaled).all()

wind_train_diff = np.max(
    np.abs(wind_train_scaled - saved_wind_train)
)

wind_test_diff = np.max(
    np.abs(wind_test_scaled - saved_wind_test)
)

# 12. Generate reproducibility hash
wind_hash = hashlib.sha256(
    wind_raw.tobytes()
).hexdigest()

# 13. Update ablation inventory
wind_record["raw"] = wind_raw
wind_record["train"] = wind_train_scaled
wind_record["test"] = wind_test_scaled
wind_record["data_hash"] = wind_hash

# 14. Final summary
print("=== CMIP6 WIND SPEED VALIDATION ===")

print("\nOriginal NetCDF:", WIND_NC_PATH.name)
print("Variable: sfcWind")
print("Units:", wind_units)

print("\nTime range:")
print("Start:", start_time)
print("End:", end_time)

print("\nSpatial locations:", location_indices)
print("Grid rows:", grid_rows.tolist())
print("Grid cols:", grid_cols.tolist())

print("\nRaw shape:", wind_raw.shape)
print("Train shape:", wind_train_scaled.shape)
print("Test shape:", wind_test_scaled.shape)

print("\nMissing values:", np.isnan(wind_raw).sum())
print("Infinite values:", np.isinf(wind_raw).sum())

print("\nMaximum Train difference:", wind_train_diff)
print("Maximum Test difference:", wind_test_diff)

print("\nRaw data SHA256:", wind_hash)

print("\nNetCDF extraction matched.")
print("Train/Test split matched.")
print("Scaler statistics matched.")
print("CMIP6 Wind preprocessing validation passed!")


=== CMIP6 WIND SPEED VALIDATION ===

Original NetCDF: Klima_Daten_10m_3h_2020_RCP26.nc
Variable: sfcWind
Units: m s-1

Time range:
Start: 2020-01-01T00:00:00.000000000
End: 2020-12-30T21:00:00.000000000

Spatial locations: [0, 5000, 10000, 15000, 20000]
Grid rows: [187, 246, 234, 222, 210]
Grid cols: [184, 202, 209, 216, 225]

Raw shape: (2920, 5)
Train shape: (2336, 5)
Test shape: (584, 5)

Missing values: 0
Infinite values: 0

Maximum Train difference: 0.0
Maximum Test difference: 0.0

Raw data SHA256: d3aeada64be9ce70ecb1034c3b7017871e3ac9c5776c547d41294424c7dd0aea

NetCDF extraction matched.
Train/Test split matched.
Scaler statistics matched.
CMIP6 Wind preprocessing validation passed!


CMIP6 PRESSURE SURFACE DATASET VALIDATION

In [14]:
import xarray as xr
import hashlib
from itertools import product

# 1. Locate original NetCDF files
PRESSURE_RAW_DIR = (
    PROJECT_ROOT / "datasets" / "climate" / "raw"
)

pressure_candidates = sorted(
    PRESSURE_RAW_DIR.glob("ps_EUR-11_*.nc")
)

if not pressure_candidates:
    raise FileNotFoundError(
        "No original pressure NetCDF file found."
    )

# 2. Load existing Pressure dataset from inventory
pressure_record = ABLATION_DATASETS["CMIP6 Pressure"]
pressure_bundle = pressure_record["metadata"]

saved_pressure = np.asarray(
    pressure_record["raw"],
    dtype=np.float32
)

assert saved_pressure.shape == (400, 5)
assert np.isfinite(saved_pressure).all()

# 3. Read saved geographic grid positions
pressure_rows = np.asarray(
    pressure_bundle["grid_rows"], dtype=int
)

pressure_cols = np.asarray(
    pressure_bundle["grid_cols"], dtype=int
)

pressure_locations = pressure_bundle["location_indices"]

assert len(pressure_rows) == 5
assert len(pressure_cols) == 5

# Verify consistency with Wind locations
np.testing.assert_array_equal(
    pressure_rows, grid_rows
)

np.testing.assert_array_equal(
    pressure_cols, grid_cols
)

assert list(pressure_locations) == list(location_indices)

# 4. Reconstruct from original NetCDF
# Find the file that matches the saved Pressure data
matching_sources = []

for nc_path in pressure_candidates:

    with xr.open_dataset(nc_path) as ds:

        if "ps" not in ds.data_vars:
            continue

        if ds.sizes["time"] < 400:
            continue

        selected = ds["ps"].isel(
            time=slice(0, 400),
            rlat=xr.DataArray(
                pressure_rows, dims="site"
            ),
            rlon=xr.DataArray(
                pressure_cols, dims="site"
            )
        )

        reconstructed = (
            selected
            .transpose("time", "site")
            .values
            .astype(np.float32)
        )

        if np.array_equal(reconstructed, saved_pressure):

            matching_sources.append({
                "path": nc_path,
                "raw": reconstructed,
                "units": ds["ps"].attrs.get(
                    "units", "unknown"
                ),
                "total_timesteps": ds.sizes["time"],
                "start_time": ds["time"].values[0],
                "end_time": ds["time"].values[399],
            })

assert len(matching_sources) == 1, (
    "Expected exactly one matching Pressure NetCDF "
    f"source, found {len(matching_sources)}."
)

source = matching_sources[0]
pressure_raw = source["raw"]

# 5. Validate reconstructed data
assert pressure_raw.shape == (400, 5)
assert np.isfinite(pressure_raw).all()

np.testing.assert_array_equal(
    pressure_raw, saved_pressure
)

pressure_diff = np.max(
    np.abs(pressure_raw - saved_pressure)
)

# 6. Validate chronological split feasibility
PRESSURE_TRAIN_RATIO = 0.8

pressure_split_idx = int(
    len(pressure_raw) * PRESSURE_TRAIN_RATIO
)

pressure_train_length = pressure_split_idx
pressure_test_length = (
    len(pressure_raw) - pressure_split_idx
)

# 7. Check all ablation grid configurations
pressure_grid_results = []

for p, h in product(PATCH_LENGTHS, HORIZONS):

    train_windows = max(
        0, pressure_train_length - p - h
    )

    test_windows = max(
        0, pressure_test_length - p - h
    )

    pressure_grid_results.append({
        "PatchLen": p,
        "Horizon": h,
        "TrainWindows": train_windows,
        "TestWindows": test_windows,
        "Feasible": (
            train_windows > 0 and test_windows > 0
        )
    })

pressure_grid_df = pd.DataFrame(
    pressure_grid_results
)

feasible_count = int(
    pressure_grid_df["Feasible"].sum()
)

assert len(pressure_grid_df) == 56

# 8. Find minimum total length for full grid
max_patch = max(PATCH_LENGTHS)
max_horizon = max(HORIZONS)

minimum_required = next(
    n for n in range(1, 10000)
    if n - int(n * PRESSURE_TRAIN_RATIO)
       > max_patch + max_horizon
)

# 9. Generate reproducibility hash
pressure_hash = hashlib.sha256(
    pressure_raw.tobytes()
).hexdigest()

# 10. Update dataset inventory
pressure_record["raw"] = pressure_raw
pressure_record["data_hash"] = pressure_hash

# Do not create or overwrite Train/Test yet.

# 11. Print validation summary
print("=== CMIP6 PRESSURE SURFACE VALIDATION ===")

print("\nOriginal NetCDF:", source["path"].name)
print("Variable: ps")
print("Units:", source["units"])

print("\nNetCDF total timestamps:",
      source["total_timesteps"])

print("Selected time range:")
print("Start:", source["start_time"])
print("End:", source["end_time"])

print("\nSpatial locations:", pressure_locations)
print("Grid rows:", pressure_rows.tolist())
print("Grid cols:", pressure_cols.tolist())

print("\nRaw shape:", pressure_raw.shape)
print("Missing values:", np.isnan(pressure_raw).sum())
print("Infinite values:", np.isinf(pressure_raw).sum())
print("Maximum source difference:", pressure_diff)

print("\nRaw SHA256:", pressure_hash)

print("\n=== ABLATION GRID FEASIBILITY ===")
print("Available timestamps:", len(pressure_raw))
print("Proposed Train length:", pressure_train_length)
print("Proposed Test length:", pressure_test_length)

print("Total (p, H) pairs:", len(pressure_grid_df))
print("Feasible pairs:", feasible_count)
print("Infeasible pairs:",
      len(pressure_grid_df) - feasible_count)

print("\nFeasible configurations:")
display(
    pressure_grid_df[
        pressure_grid_df["Feasible"]
    ]
)

print("\nMinimum timestamps for full grid:",
      minimum_required)

print("\nNetCDF extraction matched.")
print("Spatial positions matched.")
print("Pressure source validation passed!")

if feasible_count < 56:
    print(
        "\nWARNING: Current 400 timestamps are "
        "INSUFFICIENT for full ablation."
    )
    print(
        "Pressure Train/Test data have NOT been "
        "created or modified."
    )


=== CMIP6 PRESSURE SURFACE VALIDATION ===

Original NetCDF: ps_EUR-11_MPI-M-MPI-ESM-LR_rcp26_r1i1p1_CLMcom-BTU-CCLM4-8-17_v1_3hr_202001010000-202012312100.nc
Variable: ps
Units: Pa

NetCDF total timestamps: 2928
Selected time range:
Start: 2020-01-01T00:00:00.000000000
End: 2020-02-19T21:00:00.000000000

Spatial locations: [0, 5000, 10000, 15000, 20000]
Grid rows: [187, 246, 234, 222, 210]
Grid cols: [184, 202, 209, 216, 225]

Raw shape: (400, 5)
Missing values: 0
Infinite values: 0
Maximum source difference: 0.0

Raw SHA256: ef61c7daa57b3eda8d443555982cbcd6f97956a8f57d7b30fcd91325673d9108

=== ABLATION GRID FEASIBILITY ===
Available timestamps: 400
Proposed Train length: 320
Proposed Test length: 80
Total (p, H) pairs: 56
Feasible pairs: 4
Infeasible pairs: 52

Feasible configurations:


,PatchLen,Horizon,TrainWindows,TestWindows,Feasible
0,70,2,248,8,True
1,70,4,246,6,True
2,70,6,244,4,True
3,70,8,242,2,True



Minimum timestamps for full grid: 731

NetCDF extraction matched.
Spatial positions matched.
Pressure source validation passed!

Pressure Train/Test data have NOT been created or modified.


FULL PRESSURE DATASET PREPARATION

In [15]:
from sklearn.preprocessing import MinMaxScaler
from itertools import product
import hashlib

# 1. Reference length from author's pressure script
PRESSURE_TOTAL = 2500
PRESSURE_TRAIN_RATIO = 0.8

PRESSURE_PATCH_LENGTHS = [70, 80, 90, 100, 110, 120, 130]
PRESSURE_HORIZONS = [2, 4, 6, 8, 10, 12, 14, 16]

# 2. Verify source availability
pressure_source_path = source["path"]

available_timesteps = source["total_timesteps"]

print("=== FULL PRESSURE PREPARATION ===")
print("Source:", pressure_source_path.name)
print("Available timestamps:", available_timesteps)
print("Requested timestamps:", PRESSURE_TOTAL)

if available_timesteps < PRESSURE_TOTAL:
    raise ValueError(
        f"NetCDF contains only {available_timesteps} timestamps; "
        f"cannot select {PRESSURE_TOTAL}."
    )

# 3. Extract first 2500 timestamps at the same 5 sites
with xr.open_dataset(pressure_source_path) as ds:

    pressure_full_raw = (
        ds["ps"]
        .isel(
            time=slice(0, PRESSURE_TOTAL),
            rlat=xr.DataArray(pressure_rows, dims="site"),
            rlon=xr.DataArray(pressure_cols, dims="site")
        )
        .transpose("time", "site")
        .values
        .astype(np.float32)
    )

assert pressure_full_raw.shape == (2500, 5)
assert np.isfinite(pressure_full_raw).all()

# 4. Verify consistency with previous 400 timestamps
np.testing.assert_array_equal(
    pressure_full_raw[:400],
    saved_pressure
)

# 5. Chronological 80/20 split
pressure_split = int(
    PRESSURE_TOTAL * PRESSURE_TRAIN_RATIO
)

pressure_train_raw = pressure_full_raw[:pressure_split]
pressure_test_raw = pressure_full_raw[pressure_split:]

assert pressure_train_raw.shape == (2000, 5)
assert pressure_test_raw.shape == (500, 5)

# 6. Min-Max Scaling: fit on Train only
pressure_scaler = MinMaxScaler(feature_range=(0, 1))

pressure_train_scaled = pressure_scaler.fit_transform(
    pressure_train_raw
).astype(np.float32)

pressure_test_scaled = pressure_scaler.transform(
    pressure_test_raw
).astype(np.float32)

assert np.isfinite(pressure_train_scaled).all()
assert np.isfinite(pressure_test_scaled).all()

# 7. Validate complete ablation grid
pressure_full_grid = []

for p, h in product(
    PRESSURE_PATCH_LENGTHS,
    PRESSURE_HORIZONS
):

    n_train = len(pressure_train_scaled) - p - h
    n_test = len(pressure_test_scaled) - p - h

    assert n_train > 0
    assert n_test > 0

    pressure_full_grid.append({
        "PatchLen": p,
        "Horizon": h,
        "TrainWindows": n_train,
        "TestWindows": n_test
    })

pressure_full_grid_df = pd.DataFrame(
    pressure_full_grid
)

assert len(pressure_full_grid_df) == 56

# 8. Generate data hash
pressure_full_hash = hashlib.sha256(
    pressure_full_raw.tobytes()
).hexdigest()

# 9. Update in-memory inventory only
pressure_record["raw"] = pressure_full_raw
pressure_record["train"] = pressure_train_scaled
pressure_record["test"] = pressure_test_scaled
pressure_record["data_hash"] = pressure_full_hash

pressure_record["preparation_status"] = (
    "candidate_2500_not_yet_confirmed_against_ablation"
)

# 10. Print summary
print("\n=== PRESSURE TRAIN/TEST VALIDATION ===")

print("Raw:", pressure_full_raw.shape)
print("Train:", pressure_train_scaled.shape)
print("Test:", pressure_test_scaled.shape)

print("\nFirst 400 timestamps: MATCHED")
print("Missing:", np.isnan(pressure_full_raw).sum())
print("Infinite:", np.isinf(pressure_full_raw).sum())

print("\n=== ABLATION GRID ===")
print("Total configurations:", len(pressure_full_grid_df))

print("Train windows range:",
      pressure_full_grid_df["TrainWindows"].min(),
      "to",
      pressure_full_grid_df["TrainWindows"].max())

print("Test windows range:",
      pressure_full_grid_df["TestWindows"].min(),
      "to",
      pressure_full_grid_df["TestWindows"].max())

print("\nRaw SHA256:", pressure_full_hash)

print("\nAll 56 pressure configurations are feasible.")
print("No existing files were overwritten.")
print("Pressure dataset candidate prepared successfully!")


=== FULL PRESSURE PREPARATION ===
Source: ps_EUR-11_MPI-M-MPI-ESM-LR_rcp26_r1i1p1_CLMcom-BTU-CCLM4-8-17_v1_3hr_202001010000-202012312100.nc
Available timestamps: 2928
Requested timestamps: 2500

=== PRESSURE TRAIN/TEST VALIDATION ===
Raw: (2500, 5)
Train: (2000, 5)
Test: (500, 5)

First 400 timestamps: MATCHED
Missing: 0
Infinite: 0

=== ABLATION GRID ===
Total configurations: 56
Train windows range: 1854 to 1928
Test windows range: 354 to 428

Raw SHA256: ab7e5037d7cb437686e9654c79747aa90ccb27398cf8c3923b1b7ee4da2d5ded

All 56 pressure configurations are feasible.
No existing files were overwritten.
Pressure dataset candidate prepared successfully!


UNIFIED SLIDING WINDOWS VALIDATION

In [16]:
from itertools import product

# 1. Author's Ablation Grid
PATCH_LENGTHS = [70, 80, 90, 100, 110, 120, 130]
HORIZONS = [2, 4, 6, 8, 10, 12, 14, 16]

# 2. Unified sliding-window function
def build_ablation_windows(data, patch_len, horizon):
    """
    Input:
        data: (N, F)
        patch_len: number of past timestamps
        horizon: number of future timestamps

    Output:
        X: (samples, patch_len, F)
        Y: (samples, F * horizon)

    Reproduces the original author's indexing.
    """
    data = np.asarray(data, dtype=np.float32)

    if data.ndim != 2:
        raise ValueError("Expected 2D data (N, F).")

    n_samples = len(data) - patch_len - horizon

    if n_samples <= 0:
        raise ValueError(
            f"Insufficient data: N={len(data)}, "
            f"P={patch_len}, H={horizon}"
        )

    X, Y = [], []

    for i in range(n_samples):
        X.append(data[i:i + patch_len])

        Y.append(
            data[
                i + patch_len:
                i + patch_len + horizon
            ].T.reshape(-1)
        )

    return (
        np.stack(X).astype(np.float32),
        np.stack(Y).astype(np.float32)
    )


# 3. Validate all datasets and all grid configurations
validation_records = []

for name, record in ABLATION_DATASETS.items():

    train = np.asarray(record["train"], dtype=np.float32)
    test = np.asarray(record["test"], dtype=np.float32)

    assert train.ndim == 2
    assert test.ndim == 2
    assert train.shape[1] == test.shape[1]
    assert np.isfinite(train).all()
    assert np.isfinite(test).all()

    F = train.shape[1]

    for p, h in product(PATCH_LENGTHS, HORIZONS):

        n_train = len(train) - p - h
        n_test = len(test) - p - h

        assert n_train > 0
        assert n_test > 0

        validation_records.append({
            "Dataset": name,
            "Channels": F,
            "PatchLen": p,
            "Horizon": h,
            "TrainWindows": n_train,
            "TestWindows": n_test
        })

    # 4. Validate actual window contents at grid extremes
    for p, h in [(70, 2), (130, 16)]:

        for split_name, data in [
            ("Train", train),
            ("Test", test)
        ]:

            X, Y = build_ablation_windows(data, p, h)

            expected_samples = len(data) - p - h

            assert X.shape == (
                expected_samples, p, F
            )

            assert Y.shape == (
                expected_samples, F * h
            )

            # First input and target
            np.testing.assert_array_equal(
                X[0], data[:p]
            )

            np.testing.assert_array_equal(
                Y[0],
                data[p:p + h].T.reshape(-1)
            )

            # Last input and target
            last_idx = expected_samples - 1

            np.testing.assert_array_equal(
                X[-1],
                data[last_idx:last_idx + p]
            )

            np.testing.assert_array_equal(
                Y[-1],
                data[
                    last_idx + p:
                    last_idx + p + h
                ].T.reshape(-1)
            )

            assert np.isfinite(X).all()
            assert np.isfinite(Y).all()

            del X, Y

    print(
        f"{name:<16} "
        f"Train={train.shape}, "
        f"Test={test.shape} — PASSED"
    )


# 5. Build validation summary
ablation_grid_df = pd.DataFrame(validation_records)

ablation_summary = (
    ablation_grid_df
    .groupby("Dataset", sort=False)
    .agg(
        Channels=("Channels", "first"),
        Configurations=("PatchLen", "count"),
        MinTrainWindows=("TrainWindows", "min"),
        MaxTrainWindows=("TrainWindows", "max"),
        MinTestWindows=("TestWindows", "min"),
        MaxTestWindows=("TestWindows", "max")
    )
    .reset_index()
)

assert len(ablation_grid_df) == 4 * 56
assert (ablation_summary["Configurations"] == 56).all()

# 6. Final results
print("\n=== ABLATION SLIDING WINDOWS VALIDATION ===")
display(ablation_summary)

print("\nTotal datasets:", len(ABLATION_DATASETS))
print("Configurations per dataset:", 56)
print("Total dataset-grid combinations:",
      len(ablation_grid_df))

print("\nFirst and last windows validated.")
print("Input/output shapes validated.")
print("All dataset-grid combinations are feasible.")
print("Unified sliding windows validation passed!")


Electricity      Train=(2800, 6), Test=(700, 6) — PASSED
Financial        Train=(2320, 2), Test=(580, 2) — PASSED
CMIP6 Wind       Train=(2336, 5), Test=(584, 5) — PASSED
CMIP6 Pressure   Train=(2000, 5), Test=(500, 5) — PASSED

=== ABLATION SLIDING WINDOWS VALIDATION ===


,Dataset,Channels,Configurations,MinTrainWindows,MaxTrainWindows,MinTestWindows,MaxTestWindows
0,Electricity,6,56,2654,2728,554,628
1,Financial,2,56,2174,2248,434,508
2,CMIP6 Wind,5,56,2190,2264,438,512
3,CMIP6 Pressure,5,56,1854,1928,354,428



Total datasets: 4
Configurations per dataset: 56
Total dataset-grid combinations: 224

First and last windows validated.
Input/output shapes validated.
All dataset-grid combinations are feasible.
Unified sliding windows validation passed!


CELL 10 — EXPORT ABLATION DATASETS

In [17]:
import hashlib
import pandas as pd
import numpy as np
import torch
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

# 1. Output directory
ABLATION_DIR = PROJECT_ROOT / "experiments" / "05_ablation"
DATA_DIR = ABLATION_DIR / "data"

DATA_DIR.mkdir(parents=True, exist_ok=True)

# 2. Author's ablation configuration
PATCH_LENGTHS = [70, 80, 90, 100, 110, 120, 130]
HORIZONS = [2, 4, 6, 8, 10, 12, 14, 16]

# 3. Dataset filenames and provenance
DATASET_SPECS = {
    "Electricity": {
        "filename": "electricity_ablation.pt",
        "source": "datasets/electricity/energy_dataset.csv",
        "status": "verified_local_pipeline",
        "notes": "3500 timestamps, six generation channels, forward fill"
    },
    "Financial": {
        "filename": "financial_ablation.pt",
        "source": "experiments/03_financial/data/crypto_coins.csv",
        "status": "verified_local_pipeline",
        "notes": "Two consecutive BTC segments, 2900 rows each"
    },
    "CMIP6 Wind": {
        "filename": "cmip6_wind_ablation.pt",
        "source": "Klima_Daten_10m_3h_2020_RCP26.nc",
        "status": "verified_local_extraction",
        "notes": "2920 timestamps, five spatial locations"
    },
    "CMIP6 Pressure": {
        "filename": "cmip6_pressure_candidate_2500.pt",
        "source": source["path"].name,
        "status": "candidate_2500",
        "notes": (
            "2500 timestamps inspired by author's separate "
            "pressure script; exact ablation selection unconfirmed"
        )
    }
}

EXPECTED_SHAPES = {
    "Electricity": ((3500, 6), (2800, 6), (700, 6)),
    "Financial": ((2900, 2), (2320, 2), (580, 2)),
    "CMIP6 Wind": ((2920, 5), (2336, 5), (584, 5)),
    "CMIP6 Pressure": ((2500, 5), (2000, 5), (500, 5))
}

def as_numpy(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)

def sha256_arrays(*arrays):
    digest = hashlib.sha256()
    for array in arrays:
        digest.update(
            np.ascontiguousarray(array).tobytes()
        )
    return digest.hexdigest()

# 4. Export each dataset
export_records = []

for name, spec in DATASET_SPECS.items():

    record = ABLATION_DATASETS[name]

    raw = as_numpy(record["raw"]).astype(np.float32)
    train = as_numpy(record["train"]).astype(np.float32)
    test = as_numpy(record["test"]).astype(np.float32)

    expected_raw, expected_train, expected_test = (
        EXPECTED_SHAPES[name]
    )

    assert raw.shape == expected_raw
    assert train.shape == expected_train
    assert test.shape == expected_test

    assert len(train) + len(test) == len(raw)
    assert train.shape[1] == test.shape[1]

    assert np.isfinite(raw).all()
    assert np.isfinite(train).all()
    assert np.isfinite(test).all()

    # 5. Revalidate Min-Max Scaling
    scaler = MinMaxScaler()

    reconstructed_train = scaler.fit_transform(
        raw[:len(train)]
    ).astype(np.float32)

    reconstructed_test = scaler.transform(
        raw[len(train):]
    ).astype(np.float32)

    np.testing.assert_allclose(
        reconstructed_train, train,
        rtol=1e-6, atol=1e-6
    )

    np.testing.assert_allclose(
        reconstructed_test, test,
        rtol=1e-6, atol=1e-6
    )

    # 6. Dataset channels
    if name == "Electricity":
        channel_names = list(channels)

    elif name == "Financial":
        channel_names = list(
            record["metadata"]["channels"]
        )

    else:
        spatial_indices = record[
            "metadata"
        ]["location_indices"]

        channel_names = [
            f"location_{idx}"
            for idx in spatial_indices
        ]

    assert len(channel_names) == raw.shape[1]

    # 7. Reproducibility hashes
    raw_hash = sha256_arrays(raw)
    scaled_hash = sha256_arrays(train, test)

    # Cross-check against previously validated hashes
    previous_hash = record.get(
        "data_hash",
        record["metadata"].get("data_hash")
    )

    if previous_hash is not None:
        expected_hash = (
            scaled_hash if name == "Financial"
            else raw_hash
        )

        assert previous_hash == expected_hash, (
            f"Hash mismatch for {name}"
        )

    # 8. Prepare a self-contained checkpoint
    checkpoint = {
        "dataset_name": name,
        "source": spec["source"],
        "status": spec["status"],
        "notes": spec["notes"],

        "raw": torch.from_numpy(raw.copy()),
        "train": torch.from_numpy(train.copy()),
        "test": torch.from_numpy(test.copy()),

        "channels": channel_names,
        "input_dim": int(raw.shape[1]),

        "train_ratio": float(len(train) / len(raw)),
        "split_index": int(len(train)),

        "scaler_min": torch.tensor(
            scaler.data_min_, dtype=torch.float32
        ),
        "scaler_max": torch.tensor(
            scaler.data_max_, dtype=torch.float32
        ),

        "patch_lengths": PATCH_LENGTHS,
        "horizons": HORIZONS,

        "raw_sha256": raw_hash,
        "scaled_sha256": scaled_hash,
    }

    # 9. Save without overwriting a previous export
    save_path = DATA_DIR / spec["filename"]

    if save_path.exists():
        saved = torch.load(
            save_path, map_location="cpu",
            weights_only=True
        )

        assert saved["raw_sha256"] == raw_hash
        assert saved["scaled_sha256"] == scaled_hash
        assert saved["status"] == spec["status"]

        print(f"EXISTING VERIFIED: {save_path.name}")

    else:
        temp_path = save_path.with_suffix(".pt.tmp")

        torch.save(checkpoint, temp_path)
        temp_path.replace(save_path)

        print(f"SAVED: {save_path.name}")

    # 10. Verify the exported file
    loaded = torch.load(
        save_path, map_location="cpu",
        weights_only=True
    )

    assert tuple(loaded["raw"].shape) == raw.shape
    assert tuple(loaded["train"].shape) == train.shape
    assert tuple(loaded["test"].shape) == test.shape

    assert sha256_arrays(
        loaded["raw"].numpy()
    ) == raw_hash

    assert sha256_arrays(
        loaded["train"].numpy(),
        loaded["test"].numpy()
    ) == scaled_hash

    # 11. Add manifest record
    export_records.append({
        "Dataset": name,
        "Channels": raw.shape[1],
        "RawRows": len(raw),
        "TrainRows": len(train),
        "TestRows": len(test),
        "Configurations": len(PATCH_LENGTHS) * len(HORIZONS),
        "Status": spec["status"],
        "File": spec["filename"],
        "RawSHA256": raw_hash,
        "ScaledSHA256": scaled_hash
    })

# 12. Export manifest
manifest_df = pd.DataFrame(export_records)

manifest_path = DATA_DIR / "ablation_manifest.csv"

manifest_df.to_csv(
    manifest_path, index=False
)

# 13. Final validation
assert len(manifest_df) == 4
assert manifest_df["Configurations"].sum() == 224

print("\n=== ABLATION DATA EXPORT SUMMARY ===")

display(
    manifest_df[
        [
            "Dataset", "Channels",
            "RawRows", "TrainRows",
            "TestRows", "Configurations",
            "Status", "File"
        ]
    ]
)

print("\nTotal datasets:", len(manifest_df))
print("Total dataset-grid combinations:",
      manifest_df["Configurations"].sum())

print("\nOutput directory:", DATA_DIR)
print("Manifest:", manifest_path)

print("\nAll exported datasets verified!")
print("Ablation data preparation completed!")


SAVED: electricity_ablation.pt
SAVED: financial_ablation.pt
SAVED: cmip6_wind_ablation.pt
SAVED: cmip6_pressure_candidate_2500.pt

=== ABLATION DATA EXPORT SUMMARY ===


,Dataset,Channels,RawRows,TrainRows,TestRows,Configurations,Status,File
0,Electricity,6,3500,2800,700,56,verified_local_pipeline,electricity_ablation.pt
1,Financial,2,2900,2320,580,56,verified_local_pipeline,financial_ablation.pt
2,CMIP6 Wind,5,2920,2336,584,56,verified_local_extraction,cmip6_wind_ablation.pt
3,CMIP6 Pressure,5,2500,2000,500,56,candidate_2500,cmip6_pressure_candidate_2500.pt



Total datasets: 4
Total dataset-grid combinations: 224

Output directory: c:\Users\Win 11 New\Desktop\pa gk\experiments\05_ablation\data
Manifest: c:\Users\Win 11 New\Desktop\pa gk\experiments\05_ablation\data\ablation_manifest.csv

All exported datasets verified!
Ablation data preparation completed!
